### COUSP Dataset update

**Auteur** : Esteban Montandon

**Date** : Mars 2024

**Pays** : RDC

In [1]:
# !pip install --upgrade pyarrow # Environment change errors
%conda install -c conda-forge pyarrow -y -q

In [2]:
import os 
import pandas as pd
from datetime import datetime
import re
from openhexa.sdk.workspaces import workspace
from io import StringIO
import tempfile
from sqlalchemy import create_engine

import pyarrow ##

### Nom du dernier fichier téléchargé

In [3]:
# parameters
nouvelles_donnees_path = "/home/hexa/workspace/tdb-suivi-epidemio/data/raw/DSE_data_2024/2024_Database_Week32.csv"

In [4]:
# Set DB engine
engine = create_engine(os.environ["WORKSPACE_DATABASE_URL"])

In [5]:
# extrait l'année et la semaine du nom du fichier
nouvelles_fishier_nom = os.path.basename(nouvelles_donnees_path) 
nouvelle_version = nouvelles_fishier_nom.split('.')[0]
print(f"Dernier fichier téléchargé: {nouvelles_fishier_nom}")

Dernier fichier téléchargé: 2024_Database_Week32.csv


In [6]:
# Nom de la nouvelle version 
date_version = f"{nouvelle_version.lower().replace('database_','')}_{datetime.now().strftime('%Y_%m_%d_%H%M')}"
print(f"Nouvelle version: {date_version}")

Nouvelle version: 2024_week32_2024_09_05_1513


### Chargement du tableau: DSE_Surv_Epi_Complete

In [7]:
# Chargement du tableau DSE_Surv_Epi_Complete
DSE_Surv_Epi_Complete_table = pd.read_sql_table('DSE_Surv_Epi_Complete',con=engine)
dse_surv_epi_week_available = DSE_Surv_Epi_Complete_table[DSE_Surv_Epi_Complete_table['year'] == DSE_Surv_Epi_Complete_table.year.max()].NUMSEM.max()
print(f"DSE_Surv_Epi_Complete dernière semaine disponible: {DSE_Surv_Epi_Complete_table.year.unique().max()} - {dse_surv_epi_week_available}")

DSE_Surv_Epi_Complete dernière semaine disponible: 2024.0 - 34


### Chargement du tableau: DSE_Completude

In [8]:
# Chargement du tableau DSE_Completude
DSE_Completude_table = pd.read_sql_table('DSE_Completude',con=engine)
dse_completude_week_available = DSE_Completude_table[DSE_Completude_table['year'] == DSE_Completude_table.year.max()].NUMSEM.max()
print(f"DSE_Completude dernière semaine disponible: {DSE_Completude_table.year.unique().max()} - {dse_completude_week_available}")

DSE_Completude dernière semaine disponible: 2024 - 34


### Créer une nouvelle version de dataset

In [9]:
date_version

'2024_week32_2024_09_05_1513'

In [10]:
# Obtenir le dataset 
dataset = workspace.get_dataset("dse-epi-cousp-dataset-9594b9")

try:
    # Créer une nouvelle version
    version = dataset.create_version(date_version) # this might throw an exception if the name exists
except Exception as e:
    print(f"ERROR: {e}")
    version = dataset.latest_version
    print(f"La version du dataset deja existe, mise à jour de la dernière version {version.name}")
    

In [11]:
# Ajouter les fichiers au dataset

try:

    # Ajouter DSE_Surv_Epi_Complete .parquet
    with tempfile.NamedTemporaryFile(suffix=".parquet") as tmp:
        DSE_Surv_Epi_Complete_table.to_parquet(tmp.name)
        version.add_file(tmp.name, filename=f"DSE_Surv_Epi_Complete_{date_version}.parquet")


    # Ajouter DSE_Completude .parquet
    with tempfile.NamedTemporaryFile(suffix=".parquet") as tmp:
        DSE_Completude_table.to_parquet(tmp.name)
        version.add_file(tmp.name, filename=f"DSE_Completude_{date_version}.parquet")

    # To add a new file, copy-paste the lines above for the new table...
    
    print(f"Nouvelle version du dataset {date_version} créée")
    
except Exception as e:
    print(e)
    raise


Nouvelle version du dataset 2024_week32_2024_09_05_1513 créée


In [12]:
# helper openhexa code in: 
# https://github.com/BLSQ/openhexa/wiki/Using-the-OpenHEXA-SDK#working-with-datasets
